# SageMaker real-time invoke - MedGemma

Sends a chest X-ray + prompt directly to a real-time SageMaker endpoint using the OpenAI-compatible chat completions payload expected by vLLM.

In [ ]:
import base64
import json
import time
from pathlib import Path
from pprint import pprint

from boto3.session import Session

## Config & clients

In [ ]:
ENDPOINT_NAME = "rt-testjun22-v2"
MODEL_ID = "google/medgemma-1.5-4b-it"
REGION_NAME = "us-east-1"
PROFILE_NAME="sandbox"

session = Session(region_name=REGION_NAME,profile_name=PROFILE_NAME)
runtime = session.client("sagemaker-runtime")
sagemaker = session.client("sagemaker")

In [ ]:
endpoint = sagemaker.describe_endpoint(EndpointName=ENDPOINT_NAME)
print(endpoint["EndpointStatus"])
pprint(endpoint.get("ProductionVariants", []))

## Input image

In [ ]:
def find_input_image() -> Path:
    candidates = [
        Path("inputs/chest_xray.png"),
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError("Could not find chest_xray.png from this notebook working directory")


IMAGE_PATH = find_input_image()
image_b64 = base64.b64encode(IMAGE_PATH.read_bytes()).decode("ascii")
print(IMAGE_PATH)
print(f"Payload image size: {len(image_b64) / 1024 / 1024:.2f} MiB base64")

## Helpers

In [ ]:
def build_payload(prompt: str, max_tokens: int = 512) -> bytes:
    """Build an OpenAI-compatible chat-completions payload for vLLM."""
    body = {
        "model": MODEL_ID,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{image_b64}"},
                    },
                ],
            },
        ],
        "max_tokens": max_tokens,
        "temperature": 0,
    }
    return json.dumps(body).encode("utf-8")

In [ ]:
def invoke_endpoint_realtime(payload: bytes) -> tuple[dict, float]:
    """Invoke a SageMaker real-time endpoint and return parsed JSON plus latency."""
    start = time.perf_counter()
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=payload,
    )
    latency_seconds = time.perf_counter() - start
    raw_body = response["Body"].read()
    return json.loads(raw_body.decode("utf-8")), latency_seconds

In [ ]:
def extract_assistant_text(response_json: dict) -> str:
    """Extract assistant text from an OpenAI-compatible chat response."""
    choices = response_json.get("choices") or []
    if not choices:
        return ""

    message = choices[0].get("message") or {}
    content = message.get("content", "")
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(item.get("text", "") for item in content if isinstance(item, dict))
    return str(content)

## Run

## Examples

Five test prompts sent to the endpoint in a loop.

In [ ]:
EXAMPLES = [
    {
        "name": "1. General description",
        "prompt": "Describe this chest X-ray. Note any abnormal findings.",
        "max_tokens": 512,
    },
    {
        "name": "2. Structured radiology report",
        "prompt": (
            "Provide a structured radiology report for this chest X-ray "
            "with the following sections: Findings, Impression, Recommendation."
        ),
        "max_tokens": 512,
    },
    {
        "name": "3. Specific pathology check",
        "prompt": "Is there evidence of pneumonia in this chest X-ray? Explain your reasoning.",
        "max_tokens": 256,
    },
    {
        "name": "4. Severity assessment",
        "prompt": (
            "Review this chest X-ray and rate the severity of any findings "
            "as: Normal, Mild, Moderate, or Severe. Justify your rating."
        ),
        "max_tokens": 256,
    },
    {
        "name": "5. Differential diagnosis",
        "prompt": (
            "Based on this chest X-ray, list the top three possible diagnoses "
            "in order of likelihood and briefly explain each."
        ),
        "max_tokens": 512,
    },
]

results = []

for example in EXAMPLES:
    print(f"{'=' * 60}")
    print(f"Running: {example['name']}")
    print(f"Prompt : {example['prompt'][:80]}..." if len(example['prompt']) > 80 else f"Prompt : {example['prompt']}")
    print()

    payload = build_payload(example["prompt"], max_tokens=example["max_tokens"])
    response_json, latency = invoke_endpoint_realtime(payload)
    text = extract_assistant_text(response_json)
    usage = response_json.get("usage", {})

    print(f"Latency         : {latency:.1f}s")
    print(f"Tokens (in/out) : {usage.get('prompt_tokens', '?')} / {usage.get('completion_tokens', '?')}")
    print()
    print(text)
    print()

    results.append({
        "name": example["name"],
        "prompt": example["prompt"],
        "latency_s": round(latency, 2),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "response": text,
    })


## Summary table

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Example": r["name"],
        "Latency (s)": r["latency_s"],
        "Prompt tokens": r["prompt_tokens"],
        "Completion tokens": r["completion_tokens"],
        "Total tokens": (r["prompt_tokens"] or 0) + (r["completion_tokens"] or 0),
    }
    for r in results
])

display(summary)
